# Import the modules

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# Defining the functions

In [ ]:
def log(y_list):
    """Calculate the logarithm of a list.
    
    Parameters
    ----------
    y_list : list
        The y values.
    
    Returns
    -------
    list
        The y values that have been through the logarithm.
    """
    log_y = []
    for y_ind in y_list:
        if y_ind > 0:
            log_y.append(math.log(y_ind))
        else:
            log_y.append(0)
    return log_y

In [ ]:
# this functions returns a linear fit of x & y
def fit_decay(x,y, LIMX, LIMY):
    """Function does a linear fit.
    
    The linear fit is made on the probability of having a certain packing area.
    
    Parameters
    ----------
    x : numpy array
        Packing area.
    y : numpy array
        probability of having a certain packing area.
    LIMX : int
        The lowest defect area used for the fit.
    LIMY : int
        The lowest probability used for the fit
    Returns
    -------
    numpy array
        A linear fit of x and y.
    """
    # fit with defects above LIMX nm and proba > LIMY
    y = y[x >= LIMX]
    x = x[x >= LIMX]
    x = x[y >= LIMY]
    y = y[y >= LIMY]
    FIT = np.polyfit(x, log(y), 1)
    return (FIT)

In [ ]:
# this function divides the packdef_data in 3 blocks and returns a vector of 3 decays
def block_averaging(vect, nb_block):
    """Divide the packing data into n blocks.
    
    Parameters
    ----------
    vect : pandas dataframe
        The size of the packing defect.
    nb_block : int
        The number of blocks we want to have.
    
    Returns
    -------
    list
        Avector of 3 decays.
    """
    bornes = [int((len(vect)/3)*nb) for nb in range(nb_block+1)]
    decays = []
    for i in range(0,nb_block):
        subvect = vect[bornes[i]:bornes[i+1]]
        H = plt.hist(subvect, bins = np.arange(0.5, max(vect)+0.5))
        x = H[1]+0.5
        x = x[:len(x)-1]
        y = H[0]/sum(H[0])
        FIT = fit_decay(x, y, LIMX, LIMY)
        decays.append(abs(1/FIT[0]))
    return decays

# Defining the variables

In [ ]:
# prefix name
PREFIX = "dmpc"
# long prefix name
LONG_PREFIX = ("Total_"+PREFIX)
# set precision for writing packdef cosntants (nb of decimals) in the txt output file
PRECISION = 2
# lowest defect area used for the fit (we recommand not to touch to this value)
LIMX = 15
# lowest probability used for the fit (we recommand not to touch to this value)
LIMY = 1e-4

# Main code

## Initialise dataframe

In [ ]:
# Initialise a data frame to store packdef constants + errors
packdef_constants = pd.DataFrame(columns = ['deep', 'shallow', 'all'], 
                   index = ["PackDef_cst_global", "PackDef_cst_block1", "PackDef_cst_block2", 
                            "PackDef_cst_block3","PackDef_cst_all_blocks","error_all_blocks"])

## Create the plot of probability of a certain packing defect area

In [ ]:
# Now loop over the three default types
for DEFECT in ["deep","shallow","all"]:
    # load PackMem data
    filename = "~/Documents/Penetratin/DMPC/big/303K/ions_switch_0.8/05_analysis/PackMem/"+LONG_PREFIX+"_"+DEFECT+"_clean.txt"
    packdef_data = pd.read_csv(filename, header=None, delim_whitespace=True)[1]

    # Compute PackDef distributions (on the whole set)
    H = plt.hist(packdef_data, bins=np.arange(0.5,max(packdef_data)+0.5)) # missing plot=F
    # Length of the defects
    x = np.arange(0, max(packdef_data)-1)
    # nb of observations of a certain defect length
    y = H[0]/sum(H[0])
    FIT = fit_decay(x,y, LIMX, LIMY)
    fit_function = np.poly1d(FIT)

    ### Plot for defect fit
    plt.clf()
    plt.scatter(x, log(y), marker='o', facecolor="none", edgecolor="black")
    plt.xlim(-2, 102)
    plt.ylim(-10, -3.5)
    plt.yticks([math.log(1e-4), math.log(1e-3), math.log(1e-2)], labels=[str(1e-4), str(1e-3), str(1e-2)])
    plt.ylabel("Probability")
    plt.xlabel("Defect area ${A^2}$")
    plt.title(DEFECT)
    plt.axvline(LIMX, color='gray', linestyle='--')
    plt.axhline(math.log(LIMY), color='gray', linestyle='-')
    plt.plot(x, fit_function(x), color='red', label='Fit')

    global_inv_decay = abs(1/FIT[0])
    
    # print global results
    print("")
    print(f"Results on {PREFIX} for {DEFECT} defects")
    print(f"Total number of defects = {len(packdef_data)}")
    print(f"Using all data: {round(global_inv_decay, PRECISION)} A^2")
    
    # Compute PackDef distributions with block averaging method, and print results
    FITS_3blocks=block_averaging(packdef_data, 3)
    for i in range(3):
        print(f"Using block {i+1}: {round(FITS_3blocks[i],PRECISION)} A^2")
    inv_decay_block = sum(FITS_3blocks)/len(FITS_3blocks)
    error_inv_decay_block = np.std(FITS_3blocks)
    print(f"Mean +/- sd on 3 blocks: {round(inv_decay_block,PRECISION)} +/- {round(error_inv_decay_block,PRECISION)} A^2")
    
    # store all the results in the data frame
    packdef_constants[DEFECT]["PackDef_cst_global"] = global_inv_decay
    packdef_constants[DEFECT]["PackDef_cst_block1"] = FITS_3blocks[0]
    packdef_constants[DEFECT]["PackDef_cst_block2"] = FITS_3blocks[1]
    packdef_constants[DEFECT]["PackDef_cst_block3"] = FITS_3blocks[2]
    packdef_constants[DEFECT]["PackDef_cst_all_blocks"] = inv_decay_block
    packdef_constants[DEFECT]["error_all_blocks"] = error_inv_decay_block

## Create plot for the packing defect constant in different blocks for each defects

In [ ]:
# INTERMEDIATE plot: plot all packdef constants on a single barplot
# Allows to estimate the relative convergence of the simulation
errors=packdef_constants.T.error_all_blocks.to_frame('PackDef_cst_all_blocks')
packdef_constants[:][:5].T.plot.bar(color=["darkred", "firebrick", "indianred", "lightcoral", "rosybrown"], yerr=errors, capsize=3, rot=0)
plt.ylabel("Defect size constant ${A^2}$")
plt.title("Packing defect constants")

## Create plot for the packing defect constant each type of defect

In [ ]:
# FINAL PLOT
# Now we plot the final decays + errors computed with block averaging 
# (for each packdef) on a barplot
br = packdef_constants.loc["PackDef_cst_all_blocks"].plot.bar(color=["royalblue", "forestgreen", "firebrick"], rot=0, yerr=errors, capsize=3)
for p in br.patches:
    br.annotate(f'{p.get_height()/max(packdef_constants.loc["PackDef_cst_all_blocks"])*100:.1f}%', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='top', xytext=(0, 10), textcoords='offset points')
    br.annotate(f'{p.get_height():.1f} $A^2$', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='top', xytext=(0, -10), textcoords='offset points')
plt.ylabel("Defect size constant ${A^2}$")
plt.title("Packing defect constants computed by block averaging")